In [ ]:
# forbidden access (i think just set the flag upon super.init)
# because the flag prevents access of the super methods (and i don't think it's as much of a problem in this implementation)
# build dm

In [ ]:
"""
sandbox_time.ipynb

A sandbox to develop a time-resolved class.

Author: Stellina X. Ao
Created: 2026-07-07
Last Modified: 2026-07-07
Python Version: 3.11.14
"""

import numpy as np
import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
subj_id = "MR82"
sess_id = "20251027_152036"

In [ ]:
"""--------------------------------------------"""
# SANITY CHECKS
# beta weight sanity checks
# -> multiply beta weight by 1/binwidth_s and make sure identical to firing rate
"""--------------------------------------------"""

## init

In [ ]:
from sg.models import make_tre, Encoder

encoder = make_tre(Encoder, tr_type="dme")(
    subj_id,
    sess_id,
    norm=True,
    tv_keys=["response", "rewarded", "response_prev", "rewarded_prev"],
    stepsize_s=0.1,
)
encoder.verify()

# encoder_mb = make_tre(StrategyEncoder, tr_type="dme")(
#     subj_id,
#     sess_id,
#     norm=False,
#     stepsize_s=0.1,
#     strategy_filter="mb",
# )
# encoder_mb.verify()

# encoder_mf = make_tre(StrategyEncoder, tr_type="dme")(
#     subj_id,
#     sess_id,
#     norm=False,
#     stepsize_s=0.1,
#     strategy_filter="mf",
# )

In [ ]:
def _transform(robs):
    return robs.reshape(
        encoder.num_trials, encoder.num_bins, encoder.num_units
    ).transpose(1, 0, 2)

In [ ]:
_transform(encoder.robs).ndim

In [ ]:
regr = "response"
idxs = [encoder.dm_idxs[f"{regr}_{i}"] for i in range(15)]
encoder.encoder_weights[:, idxs]

In [ ]:
A = np.arange(21).reshape(7, 3)
A, np.hstack(A.T)

In [ ]:
np.where(encoder.tvs[4 :: encoder.num_bins, encoder.dm_idxs["rewarded_14"]] == 1)[
    0
].shape

In [ ]:
encoder.dm_idxs["response_0"]

In [ ]:
plt.figure()
plt.imshow(encoder.tvs, aspect="auto", interpolation="none")
plt.show()

In [ ]:
np.where(encoder.tvs[::15, encoder.dm_idxs["rewarded_0"] - 5] == 1)[0].shape
177 / 203 == (encoder.trial_data.rewarded == 1).mean()

In [ ]:
idxs = [encoder.dm_idxs[f"response_{i}"] - 5 for i in range(15)]
plt.figure()
plt.imshow(encoder.tvs[:45, idxs])
plt.show()

In [ ]:
plt.figure()
plt.plot(encoder.tvs[:, encoder.dm_idxs["rewarded_0"] - 5])

In [ ]:
encoder.encoder_weights

In [ ]:
from copy import deepcopy
from core.data import get_tavg_sc_cond, tv_vals

sc_tavg = {}
regr = "rewarded"

bweight_robs = {val: [] for val in tv_vals[regr]}

# get baseline explained variance
robs_baseline = encoder.robs_predict["baseline"]
encoder.robs_baseline = robs_baseline

# get tv explained variance (besides pivot)
for i in range(15):
    pivot_idx = encoder.dm_idxs[f"{regr}_{i}"] - encoder.num_tents
    encoder.tv_pivot_ko = deepcopy(encoder.tvs)
    encoder.tv_pivot_ko[:, pivot_idx] = 0

    robs_tv = encoder.encoder.predict(encoder.tv_pivot_ko)
    encoder.robs_to_subtract = robs_baseline + robs_tv

    a = get_tavg_sc_cond(
        _transform(encoder.robs)[i],  # think and you will know that this is correct
        encoder.trial_data,
        regr=regr,
        robs_to_subtract=_transform(encoder.robs_to_subtract)[i],
        subtract_robs=True,
    )

    for k in a.keys():
        bweight_robs[k].extend(a[k])

bweight_robs = {k: np.array(v) for (k, v) in bweight_robs.items()}

if regr == "response":
    pos = "left"
    neg = "right"
else:
    pos = "corr"
    neg = "incorr"
bweight_robs_ = (bweight_robs[pos] - bweight_robs[neg]) / 2

idxs = [encoder.dm_idxs[f"{regr}_{i}"] for i in range(15)]
bweight_enc_ = np.hstack(encoder.encoder_weights[:, idxs].T)

fig, ax = plt.subplots()
ax.scatter(
    bweight_robs_,
    bweight_enc_,
    s=0.5,
    alpha=0.5,
    vmin=1e-5,
    vmax=1e5,
    c=np.tile(encoder.encoder.alpha_, encoder.num_bins),
    cmap="viridis",
    norm="log",
)

In [ ]:
# plot_sctavg_weights
# check strategy encoder too
# add interaction terms and trials from block switch

In [ ]:
encoder.view_weights()

In [ ]:
encoder.view_peths()

In [ ]:
encoder.view_weights(regr="response", peth_mode="response")

In [ ]:
bins = 2
trials = 3
neurons = 4

X = np.arange(24).reshape(bins, trials, neurons)  # 2 bins, 3 trials, 4 neurons
Y = X.transpose(1, 0, 2).reshape(bins * trials, neurons)
Z = Y.reshape(trials, bins, neurons).transpose(1, 0, 2)
assert np.all(X == Z)
X, Y, Z

In [ ]:
from copy import deepcopy
from core.viz import plot_scatter

thing = "response_0"

idx = encoder.dm_idxs[thing] - encoder.num_tents
tv_ko = deepcopy(encoder.tvs)
tv_ko[:, idx] = 0

robs_predict = encoder.encoder.predict(tv_ko)
robs_2_subtract = encoder.robs_predict["baseline"] + robs_predict
robs_corrected = encoder.robs - robs_2_subtract

corr_mask = encoder.trial_data.rewarded == 1
incorr_mask = encoder.trial_data.rewarded == 0

robs_corrected_3d = robs_corrected.reshape(
    encoder.num_trials, encoder.num_bins, encoder.num_units
).transpose(1, 0, 2)

robs_corr = robs_corrected_3d[:, corr_mask, :].mean(axis=(0, 1))
robs_incorr = robs_corrected_3d[:, incorr_mask, :].mean(axis=(0, 1))

bweight_robs = (robs_corr - robs_incorr) / 2

for i in range(80):
    bweight = encoder.encoder_weights[:, encoder.dm_idxs[thing]]

plot_scatter(bweight_robs, bweight)

In [ ]:
import numpy as np

for key in encoder.tv_keys:
    print(key, np.unique(encoder.trial_data[key]))

## the t-population

In [ ]:
# select the neurons that lie along the axis
# plot their encoding for mb/mf
# is it just a different encoding pattern (i.e., they encode different things)
# or are they silent(er) in another strategy